# Notebook 8: Ghost Populationer

En *ghost population* er en uobserveret population, der kobler sig til de observerede populationer via migration, men hvorfra ingen sekvenser er samplet. Ghost populationer er biologisk relevante, fordi genomdata aldrig dækker alle eksisterende populationer: visse grupper kan være uddøde, geografisk utilgængelige eller simpelthen ikke inkluderet i studiet.

For bavianerne er dette særligt relevant: Sørensen et al. (2023) finder tegn på **tre-vejs admixture** i western yellow-populationerne ved Mahale og Katavi, et signal der ikke forklares fuldt ud af two-island eller IM-modellen alene. En ghost population kan fungere som en proxy for en uobserveret tredje linje.

**Det centrale spørgsmål:** Hvad sker der med inferensen af $N$ og $m$, når data er genereret under en model med en ghost population, men vi fitter en simplere model uden den?

In [ ]:
# Importer nødvendige pakker
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=1"

from phasic import (
    Graph, with_ipv,
    GaussPrior, HalfCauchyPrior, DataPrior,
    Adam, ExpStepSize, ExpRegularization,
    StateIndexer, Property,
    clear_caches,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from functools import partial
from itertools import combinations_with_replacement
all_pairs = partial(combinations_with_replacement, r=2)

%config InlineBackend.figure_format = 'svg'
np.random.seed(42)
sns.set_palette('tab10')
plt.rcParams['figure.figsize'] = (10, 4)

## Ghost-modellens struktur

Ghost-modellen er en **three-island model** med symmetrisk migration, hvor population 3 er *ghost*: ingen linjer samples derfra. Tilstandsrummet beskriver, hvor de observerede linjer befinder sig, men ghost-populationen kan påvirke koalescenstiderne ved at tiltrække linjer og forsinke koalescens.

**Parametre:**
- $\theta_0 = 1/N$: koalescensrate (ens for alle tre populationer for simplicitets skyld)
- $\theta_1 = m_{\text{obs}}$: migrationsrate mellem de to observerede populationer (pop 1 ↔ pop 2)
- $\theta_2 = m_{\text{ghost}}$: migrationsrate fra observerede populationer til ghost (pop 1 ↔ ghost og pop 2 ↔ ghost)

**Starttilstand:** $n/2$ linjer i population 1, $n/2$ linjer i population 2, 0 i ghost.

Sammenlignet med two-island modellen (NB3) tilføjer ghost-populationen et ekstra "reservoar" som linjer kan vandre ud i og trækker koalescens over i en uobserveret kanal.

In [ ]:
nr_samples = 4  # 2 linjer i pop1, 2 linjer i pop2

# StateIndexer: ton x pop
# pop = 1 (observeret), 2 (observeret), 3 (ghost)
indexer = StateIndexer(
    lineages=[
        Property('ton',  min_value=1, max_value=nr_samples),
        Property('pop',  min_value=1, max_value=3),
    ]
)

# Starttilstand: 2 singletons i pop1, 2 singletons i pop2, 0 i ghost
initial = [0] * indexer.state_length
initial[indexer.lineages.props_to_index(ton=1, pop=1)] = 2
initial[indexer.lineages.props_to_index(ton=1, pop=2)] = 2
# Ghost (pop=3): ingen samples

print("Starttilstand:", initial)
print("Tilstandslængde:", indexer.state_length)

In [ ]:
# Ghost-model callback
# theta = [1/N, m_obs, m_ghost]
# theta[0]: koalescensrate (alle pop)
# theta[1]: migrationsrate pop1 <-> pop2
# theta[2]: migrationsrate pop1 <-> ghost OG pop2 <-> ghost (symmetrisk)

@with_ipv(initial)
def ghost_model(state):
    transitions = []
    total_lineages = sum(state[i] for i in range(indexer.lineages.size))
    if total_lineages <= 1:
        return transitions

    # Koalescens: inden for samme population
    for i, j in all_pairs(indexer.lineages):
        pi = indexer.lineages.index_to_props(i)
        pj = indexer.lineages.index_to_props(j)
        if pi.pop != pj.pop:
            continue
        same = int(pi.ton == pj.ton)
        if same and state[i] < 2:
            continue
        if not same and (state[i] < 1 or state[j] < 1):
            continue
        new = state.copy()
        new[i] -= 1
        new[j] -= 1
        k = indexer.lineages.props_to_index(ton=pi.ton + pj.ton, pop=pi.pop)
        new[k] += 1
        coeff = np.zeros(3)  # [1/N, m_obs, m_ghost]
        coeff[0] = state[i] * (state[j] - same) / (1 + same)
        transitions.append([new, coeff])

    # Migration
    for i in range(indexer.lineages.size):
        if state[i] < 1:
            continue
        pi = indexer.lineages.index_to_props(i)

        # Bestem destination og hvilken rate der gælder
        migration_targets = []
        if pi.pop == 1:
            migration_targets = [(2, 1), (3, 2)]  # (destination_pop, theta_idx)
        elif pi.pop == 2:
            migration_targets = [(1, 1), (3, 2)]
        elif pi.pop == 3:  # ghost sender linjer tilbage
            migration_targets = [(1, 2), (2, 2)]

        for dest_pop, theta_idx in migration_targets:
            j = indexer.lineages.props_to_index(ton=pi.ton, pop=dest_pop)
            new = state.copy()
            new[i] -= 1
            new[j] += 1
            coeff = np.zeros(3)
            coeff[theta_idx] = state[i]
            transitions.append([new, coeff])

    return transitions


graph_ghost = Graph(ghost_model)
print(f"Ghost-model: {graph_ghost.vertices_length()} tilstande")

In [ ]:
# Referenceparametre
# N=1 (koalescensrate=1), m_obs=1, m_ghost=1
theta_ref = [1.0, 1.0, 1.0]
graph_ghost.update_weights(theta_ref)

print(f"E[TMRCA] = {graph_ghost.expectation():.4f}")
print(f"Var[TMRCA] = {graph_ghost.variance():.4f}")

# Sammenlign med two-island (ingen ghost, m_ghost=0)
theta_no_ghost = [1.0, 1.0, 0.0]
graph_ghost.update_weights(theta_no_ghost)
print(f"\nUden ghost (m_ghost=0):")
print(f"E[TMRCA] = {graph_ghost.expectation():.4f}")
print(f"Var[TMRCA] = {graph_ghost.variance():.4f}")

## E[TMRCA] og Var[TMRCA]: Ghost-migrationens effekt

Ghost-migrationen fungerer som en ekstra ventil, der sender linjer ud i en uobserveret population. Linjer i ghost-populationen kan ikke koalescere med hinanden (observerede linjer mødes aldrig i ghost), så ghost-migration **forsinker** koalescens ved at trække linjer ud af kontakt med hinanden.

### Hypotese
- E[TMRCA] er **voksende** i $m_{\text{ghost}}$ for fast $m_{\text{obs}}$: jo mere ghost-migration, jo længere venter linjerne i ghost-populationen og jo sjældnere mødes de. Effekten er størst ved lav $m_{\text{obs}}$, fordi ghost-kanalen da dominerer den samlede genealogi.
- Var[TMRCA] vokser endnu hurtigere med $m_{\text{ghost}}$ end E[TMRCA], fordi variansen i opholdstiden i ghost-populationen akkumuleres.

In [ ]:
m_ghost_values = np.logspace(-2, 1.5, 40)

scenarios_mobs = [
    {'label': 'Lav obs. migration ($m_{\\mathrm{obs}}=0.2$)',     'm_obs': 0.2},
    {'label': 'Moderat obs. migration ($m_{\\mathrm{obs}}=1.0$)', 'm_obs': 1.0},
    {'label': 'Høj obs. migration ($m_{\\mathrm{obs}}=5.0$)',     'm_obs': 5.0},
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for sc in scenarios_mobs:
    E_vals, Var_vals = [], []
    for m_g in m_ghost_values:
        graph_ghost.update_weights([1.0, sc['m_obs'], m_g])
        E_vals.append(graph_ghost.expectation())
        Var_vals.append(graph_ghost.variance())
    axes[0].semilogx(m_ghost_values, E_vals,   label=sc['label'])
    axes[1].semilogx(m_ghost_values, Var_vals, label=sc['label'])

# Reference: ingen ghost
for sc in scenarios_mobs:
    graph_ghost.update_weights([1.0, sc['m_obs'], 0.0])
    axes[0].axhline(graph_ghost.expectation(), linestyle=':', alpha=0.4)
    axes[1].axhline(graph_ghost.variance(),    linestyle=':', alpha=0.4)

for ax in axes:
    ax.set_xlabel('Ghost-migrationsrate $m_{\\mathrm{ghost}}$ (log-skala)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

axes[0].set_ylabel('$E[T_{\\mathrm{MRCA}}]$')
axes[1].set_ylabel('$\\mathrm{Var}[T_{\\mathrm{MRCA}}]$')
axes[0].set_title('Forventet TMRCA vs. ghost-migration')
axes[1].set_title('Varians af TMRCA vs. ghost-migration')
plt.suptitle('Ghost-migrationens effekt på koalescenstiden (stiplede linjer = ingen ghost)', fontsize=11)
plt.tight_layout()
plt.show()

## Forventet sojourn tid: Hvor lang tid tilbringes i ghost-populationen?

En central størrelse er den forventede tid linjerne tilbringer i ghost-populationen. Denne tid er direkte uobservabel, men dens størrelse bestemmer, hvor meget ghost-migrationen forsinker koalescens.

Jeg opdeler de forventede sojourn-tider i:
- Tid tilbragt med linjer i observerede populationer (pop 1 og 2)
- Tid tilbragt med mindst én linje i ghost-populationen (pop 3)

### Hypotese
- Andelen af genealogien tilbragt i ghost-populationen er proportional med $m_{\text{ghost}} / (m_{\text{ghost}} + m_{\text{obs}} + 1/N)$. For $m_{\text{ghost}} > m_{\text{obs}}$ dominerer ghost-opholdstiden.

In [ ]:
m_ghost_sweep = np.logspace(-2, 1.5, 35)
m_obs_fixed = 1.0

rows_sojourn = []
for m_g in m_ghost_sweep:
    graph_ghost.update_weights([1.0, m_obs_fixed, m_g])
    sojourn = graph_ghost.expected_sojourn_time()
    states  = graph_ghost.states()

    T_obs   = 0.0  # tid uden linjer i ghost
    T_ghost = 0.0  # tid med mindst én linje i ghost

    for s, t in zip(states, sojourn):
        # Find om nogen linje er i pop=3 (ghost)
        ghost_present = any(
            s[indexer.lineages.props_to_index(ton=ton, pop=3)] > 0
            for ton in range(1, nr_samples + 1)
            if indexer.lineages.props_to_index(ton=ton, pop=3) < len(s)
        )
        if ghost_present:
            T_ghost += t
        else:
            T_obs += t

    rows_sojourn.append({
        'm_ghost': m_g,
        'T_observeret': T_obs,
        'T_ghost': T_ghost,
        'Total': T_obs + T_ghost,
        'Ghost-andel': T_ghost / (T_obs + T_ghost) if (T_obs + T_ghost) > 0 else 0,
    })

df_soj = pd.DataFrame(rows_sojourn)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].semilogx(df_soj['m_ghost'], df_soj['T_observeret'], label='Observerede pop. (pop 1+2)')
axes[0].semilogx(df_soj['m_ghost'], df_soj['T_ghost'],      label='Ghost pop. (pop 3)')
axes[0].semilogx(df_soj['m_ghost'], df_soj['Total'],        label='Total E[TMRCA]', linestyle='--', color='k')
axes[0].set_xlabel('$m_{\\mathrm{ghost}}$ (log-skala)')
axes[0].set_ylabel('Forventet tid (koalescent-enheder)')
axes[0].set_title('Sojourn-tid opdelt på observerede vs. ghost')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].semilogx(df_soj['m_ghost'], df_soj['Ghost-andel'] * 100, color='C2')
axes[1].set_xlabel('$m_{\\mathrm{ghost}}$ (log-skala)')
axes[1].set_ylabel('Ghost-andel af total genealogi (%)')
axes[1].set_title('Andel af genealogi tilbragt i ghost-populationen')
axes[1].grid(alpha=0.3)
axes[1].axhline(50, color='red', linestyle=':', alpha=0.5, label='50%')
axes[1].legend()

plt.suptitle(f'Sojourn-tider i ghost-modellen ($m_{{\\mathrm{{obs}}}}={m_obs_fixed}$, $N=1$)', fontsize=11)
plt.tight_layout()
plt.show()

## SFS under ghost-modellen

SFS'en afspejler fordelingen af grenlængder med $k$ efterkommere. Ghost-migrationen påvirker SFS på en karakteristisk måde: fordi linjer periodisk trækkes ud i ghost og returnerer, forlænges de tidlige faser af koalescenten (mange linjer), hvilket løfter singleton-komponenterne relativt til doubleton.

### Hypotese
- Ghost-migration med $m_{\text{ghost}} > 0$ hæver den forventede singleton-grenlængde relativt til doubleton og tripleton. SFS'en bliver mere "ekstern-tung" (front-loaded) jo større ghost-migrationen er, fordi linjerne tilbringer mere tid spredt i de tidlige koalescent-faser.

In [ ]:
# Beregn forventet SFS som funktion af m_ghost
# Reward: antal linjer med præcis k efterkommere i observerede populationer (pop 1+2)

def ghost_sfs_reward(k, states, indexer, nr_samples):
    """Reward-vektor: antal linjer med præcis k efterkommere i observerede pop."""
    reward = np.zeros(len(states))
    for s_idx, s in enumerate(states):
        for pop in [1, 2]:  # kun observerede populationer
            idx = indexer.lineages.props_to_index(ton=k, pop=pop)
            if idx < len(s):
                reward[s_idx] += s[idx]
    return reward

m_ghost_sfs = [0.0, 0.5, 1.0, 3.0]
m_obs_sfs   = 1.0
mu_sfs      = 1.0

states_ghost = graph_ghost.states()
ks = list(range(1, nr_samples))  # 1-ton til (n-1)-ton

fig, ax = plt.subplots(figsize=(8, 5))

for m_g in m_ghost_sfs:
    graph_ghost.update_weights([1.0, m_obs_sfs, m_g])
    U = graph_ghost.green_matrix()
    alpha = graph_ghost.initial_distribution()

    sfs_vals = []
    for k in ks:
        r_k = ghost_sfs_reward(k, states_ghost, indexer, nr_samples)
        E_Yk = float(alpha @ U @ r_k)
        sfs_vals.append(mu_sfs * E_Yk)

    label = f'$m_{{\\mathrm{{ghost}}}}={m_g}$' + (' (ingen ghost)' if m_g == 0.0 else '')
    ax.plot(ks, sfs_vals, 'o-', label=label)

ax.set_xlabel('$k$ (antal kopier med mutation)')
ax.set_ylabel('$E[\\xi_k]$ (forventet grenlængde $\\times \\mu$)')
ax.set_xticks(ks)
ax.set_xticklabels([f'{k}-ton' for k in ks])
ax.set_title(f'Forventet SFS under ghost-modellen ($n={nr_samples}$, $m_{{\\mathrm{{obs}}}}={m_obs_sfs}$)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Ghost-modellen vs. two-island modellen: Marginal ækvivalens

Det vigtigste spørgsmål er: **kan en two-island model forveksles med en ghost-model?** Dvs. for hvilke parametre producerer ghost-modellen det samme E[TMRCA] og Var[TMRCA] som en two-island model?

Hvis E og Var er ens, er de to modeller **marginalt ækvivalente** for observationer af TMRCA-fordelingen. De to første momenter kan da ikke skelne dem.

### Hypotese
- For hvert sæt ghost-parametre $(N, m_{\text{obs}}, m_{\text{ghost}})$ eksisterer der en effektiv two-island model med parametre $(N_{\text{eff}}, m_{\text{eff}})$, der matcher E[TMRCA] nøjagtigt, men ikke Var[TMRCA]. Var[TMRCA] er dermed det diskriminerende moment.

In [ ]:
# Byg two-island graf til sammenligning (fra NB3-mønsteret)
nr_samples_2 = 2

indexer_2isl = StateIndexer(
    lineages=[
        Property('ton', min_value=1, max_value=nr_samples_2),
        Property('pop', min_value=1, max_value=2),
    ]
)

initial_2isl = [0] * indexer_2isl.state_length
initial_2isl[indexer_2isl.lineages.props_to_index(ton=1, pop=1)] = 1
initial_2isl[indexer_2isl.lineages.props_to_index(ton=1, pop=2)] = 1

@with_ipv(initial_2isl)
def two_island_ref(state):
    transitions = []
    for i, j in all_pairs(indexer_2isl.lineages):
        pi = indexer_2isl.lineages.index_to_props(i)
        pj = indexer_2isl.lineages.index_to_props(j)
        if pi.pop == pj.pop:
            same = int(pi.ton == pj.ton)
            if same and state[i] < 2: continue
            if not same and (state[i] < 1 or state[j] < 1): continue
            new = state.copy()
            new[i] -= 1; new[j] -= 1
            k = indexer_2isl.lineages.props_to_index(ton=pi.ton + pj.ton, pop=pi.pop)
            new[k] += 1
            coeff = np.zeros(2)
            coeff[0] = state[i] * (state[j] - same) / (1 + same)
            transitions.append([new, coeff])
    for i in range(indexer_2isl.lineages.size):
        if state[i] < 1: continue
        pi = indexer_2isl.lineages.index_to_props(i)
        other_pop = 2 if pi.pop == 1 else 1
        j = indexer_2isl.lineages.props_to_index(ton=pi.ton, pop=other_pop)
        new = state.copy()
        new[i] -= 1; new[j] += 1
        coeff = np.zeros(2)
        coeff[1] = state[i]
        transitions.append([new, coeff])
    return transitions

graph_2isl = Graph(two_island_ref)
print(f"Two-island: {graph_2isl.vertices_length()} tilstande")

In [ ]:
# Bygger n=2 ghost-model til direkte sammenligning
nr_samples_g2 = 2
indexer_g2 = StateIndexer(
    lineages=[
        Property('ton', min_value=1, max_value=nr_samples_g2),
        Property('pop', min_value=1, max_value=3),
    ]
)
initial_g2 = [0] * indexer_g2.state_length
initial_g2[indexer_g2.lineages.props_to_index(ton=1, pop=1)] = 1
initial_g2[indexer_g2.lineages.props_to_index(ton=1, pop=2)] = 1

@with_ipv(initial_g2)
def ghost_model_n2(state):
    transitions = []
    if sum(state[i] for i in range(indexer_g2.lineages.size)) <= 1:
        return transitions
    for i, j in all_pairs(indexer_g2.lineages):
        pi = indexer_g2.lineages.index_to_props(i)
        pj = indexer_g2.lineages.index_to_props(j)
        if pi.pop != pj.pop: continue
        same = int(pi.ton == pj.ton)
        if same and state[i] < 2: continue
        if not same and (state[i] < 1 or state[j] < 1): continue
        new = state.copy()
        new[i] -= 1; new[j] -= 1
        k = indexer_g2.lineages.props_to_index(ton=pi.ton + pj.ton, pop=pi.pop)
        new[k] += 1
        coeff = np.zeros(3)
        coeff[0] = state[i] * (state[j] - same) / (1 + same)
        transitions.append([new, coeff])
    for i in range(indexer_g2.lineages.size):
        if state[i] < 1: continue
        pi = indexer_g2.lineages.index_to_props(i)
        if pi.pop == 1:   targets = [(2, 1), (3, 2)]
        elif pi.pop == 2: targets = [(1, 1), (3, 2)]
        else:             targets = [(1, 2), (2, 2)]
        for dest_pop, theta_idx in targets:
            j = indexer_g2.lineages.props_to_index(ton=pi.ton, pop=dest_pop)
            new = state.copy()
            new[i] -= 1; new[j] += 1
            coeff = np.zeros(3)
            coeff[theta_idx] = state[i]
            transitions.append([new, coeff])
    return transitions

graph_g2 = Graph(ghost_model_n2)
print(f"Ghost n=2: {graph_g2.vertices_length()} tilstande")

# Sammenlign E og Var for ghost vs two-island ved varierende m_ghost
m_ghost_cmp = np.logspace(-2, 1.5, 40)
m_obs_cmp   = 1.0

E_ghost_cmp, Var_ghost_cmp = [], []
E_2isl_cmp,  Var_2isl_cmp  = [], []

for m_g in m_ghost_cmp:
    graph_g2.update_weights([1.0, m_obs_cmp, m_g])
    E_ghost_cmp.append(graph_g2.expectation())
    Var_ghost_cmp.append(graph_g2.variance())

# Two-island: scan m_eff så E matcher ghost
from scipy.optimize import brentq

m_eff_vals = []
Var_2isl_match = []
for E_target in E_ghost_cmp:
    def objective(m_eff):
        graph_2isl.update_weights([1.0, m_eff])
        return graph_2isl.expectation() - E_target
    try:
        m_eff = brentq(objective, 1e-3, 50.0)
        m_eff_vals.append(m_eff)
        graph_2isl.update_weights([1.0, m_eff])
        Var_2isl_match.append(graph_2isl.variance())
    except Exception:
        m_eff_vals.append(np.nan)
        Var_2isl_match.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].semilogx(m_ghost_cmp, E_ghost_cmp, label='Ghost-model $E[T]$', lw=2)
axes[0].semilogx(m_ghost_cmp, E_ghost_cmp, 'k--', alpha=0.3, label='Two-island match (per definition)')
axes[0].set_xlabel('$m_{\\mathrm{ghost}}$'); axes[0].set_ylabel('$E[T_{\\mathrm{MRCA}}]$')
axes[0].set_title('E[TMRCA]: ghost-model')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].semilogx(m_ghost_cmp, Var_ghost_cmp, label='Ghost-model $\\mathrm{Var}[T]$', lw=2)
axes[1].semilogx(m_ghost_cmp, Var_2isl_match, 'C1--', lw=2,
                 label='Two-island med matchende $E[T]$')
axes[1].set_xlabel('$m_{\\mathrm{ghost}}$'); axes[1].set_ylabel('$\\mathrm{Var}[T_{\\mathrm{MRCA}}]$')
axes[1].set_title('Var[TMRCA]: ghost vs. to-ø med samme E[T]')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Marginal ækvivalens: to-ø kan matche E[T] men ikke Var[T]', fontsize=11)
plt.tight_layout()
plt.show()

print("\nForskel i Var[TMRCA] (ghost - two-island med matched E):")
diffs = np.array(Var_ghost_cmp) - np.array(Var_2isl_match)
for mg, d in zip(m_ghost_cmp[::8], diffs[::8]):
    print(f"  m_ghost={mg:.3f}: ΔVar = {d:.4f}")

## Model-misspecifikation: Bias i inferensen

Nu simulerer jeg data under ghost-modellen og fitter en two-island model. Dette er den situation man befinder sig i i praksis: man ved ikke, om der eksisterer en ghost population, og man bruger den simplere model.

Jeg undersøger, hvad der sker med estimaterne af $N$ og $m_{\text{obs}}$, når $m_{\text{ghost}}$ varierer.

### Hypotese
- To-ø modellen overvurderer $N$ (undervurderer $1/N$) systematisk, fordi ghost-migrationen forsinker koalescens og giver en for lang observeret TMRCA. Effekten er størst ved høj $m_{\text{ghost}}$.
- To-ø modellen undervurderer $m_{\text{obs}}$, fordi dele af det observerede migrationssignal i virkeligheden skyldes ghost-kanalen.

In [ ]:
# Sand two-island model til fitting
# (n=2, theta=[1/N, m_obs])

true_N    = 1.0   # fast
true_mobs = 1.0   # fast
N_OBS     = 3000

m_ghost_misspec = [0.0, 0.2, 0.5, 1.0, 2.0, 4.0]
step_ms = ExpStepSize(first_step=0.05, last_step=0.005, tau=40.0)

rows_misspec = []
for m_g in m_ghost_misspec:
    # Simuler data fra ghost-model
    graph_g2.update_weights([1.0/true_N, true_mobs, m_g])
    obs_ghost = graph_g2.sample(N_OBS)

    # Fit two-island model
    sv = graph_2isl.svgd(
        observed_data=obs_ghost,
        prior=[
            GaussPrior(ci=[0.2, 3.0]),   # 1/N
            GaussPrior(ci=[0.1, 5.0]),   # m_obs
        ],
        n_particles=30,
        n_iter=200,
        step_size=step_ms,
    )

    rows_misspec.append({
        'm_ghost (sand)': m_g,
        'fit 1/N mean':   sv.mean()[0],
        'fit 1/N std':    sv.std()[0],
        'fit m_obs mean': sv.mean()[1],
        'fit m_obs std':  sv.std()[1],
    })
    print(f"m_ghost={m_g:.1f}: "
          f"1/N={sv.mean()[0]:.3f}±{sv.std()[0]:.3f} (sand={1/true_N:.1f}), "
          f"m_obs={sv.mean()[1]:.3f}±{sv.std()[1]:.3f} (sand={true_mobs:.1f})")

df_ms = pd.DataFrame(rows_misspec)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].errorbar(df_ms['m_ghost (sand)'], df_ms['fit 1/N mean'],
                 yerr=df_ms['fit 1/N std'] * 1.96, fmt='o-', capsize=5)
axes[0].axhline(1.0 / true_N, color='red', linestyle='--', label='Sand $1/N$')
axes[0].set_xlabel('Sand $m_{\\mathrm{ghost}}$')
axes[0].set_ylabel('Fittet $1/N$ (two-island)')
axes[0].set_title('Bias i $1/N$ ved ignoreret ghost')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].errorbar(df_ms['m_ghost (sand)'], df_ms['fit m_obs mean'],
                 yerr=df_ms['fit m_obs std'] * 1.96, fmt='s-', capsize=5, color='C1')
axes[1].axhline(true_mobs, color='red', linestyle='--', label='Sand $m_{\\mathrm{obs}}$')
axes[1].set_xlabel('Sand $m_{\\mathrm{ghost}}$')
axes[1].set_ylabel('Fittet $m_{\\mathrm{obs}}$ (two-island)')
axes[1].set_title('Bias i $m_{\\mathrm{obs}}$ ved ignoreret ghost')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Model-misspecifikation: two-island fittet på ghost-genereret data', fontsize=11)
plt.tight_layout()
plt.show()

## Kan ghost-migrationen detekteres direkte?

Jeg udvider SVGD-inferensen til den **fulde ghost-model** og undersøger, om $m_{\text{ghost}}$ kan estimeres fra data – altså om ghost-populationen er identifikabel.

Det centrale problem er, at ghost-populationen er uobserveret. Koalescenstider fra observerede populationer indeholder indirekte information om $m_{\text{ghost}}$ via:
1. Den samlede E[TMRCA] er større end to-ø-modellen forudsiger
2. Var[TMRCA] er systematisk afvigende (som vist ovenfor)

### Hypotese
- $m_{\text{ghost}}$ er identifikabelt fra TMRCA-observationer, men med markant bredere posterior end $m_{\text{obs}}$. For $m_{\text{ghost}} < 0.3$ er posterioren nær-identisk med prioren, dvs. lav ghost-migration er ikke detektabel.

In [ ]:
# SVGD på den fulde ghost-model: estimer [1/N, m_obs, m_ghost]
true_theta_ghost_full = [1.0, 1.0, 1.5]  # [1/N, m_obs, m_ghost]
N_OBS_full = 3000

graph_g2.update_weights(true_theta_ghost_full)
obs_full = graph_g2.sample(N_OBS_full)

step_full = ExpStepSize(first_step=0.05, last_step=0.005, tau=40.0)

sv_full = graph_g2.svgd(
    observed_data=obs_full,
    prior=[
        GaussPrior(ci=[0.2, 3.0]),   # 1/N
        GaussPrior(ci=[0.1, 5.0]),   # m_obs
        GaussPrior(ci=[0.0, 5.0]),   # m_ghost
    ],
    n_particles=30,
    n_iter=250,
    step_size=step_full,
)

print("=== Fuld ghost-model SVGD ===")
param_labels = ['1/N', 'm_obs', 'm_ghost']
for lbl, mean, std, true in zip(param_labels, sv_full.mean(), sv_full.std(), true_theta_ghost_full):
    print(f"  {lbl}: estimat={mean:.3f} ± {std:.3f}  (sand={true})")

sv_full.plot_ci(true_theta=true_theta_ghost_full)
plt.title(f'SVGD posterior: fuld ghost-model ($n={N_OBS_full}$ obs.)')
plt.tight_layout()
plt.show()

In [ ]:
# Pairwise posterior: korrelation mellem m_obs og m_ghost
sv_full.plot_pairwise(true_theta=true_theta_ghost_full)
plt.suptitle('Joint posterior: $(1/N,\\, m_{\\mathrm{obs}},\\, m_{\\mathrm{ghost}})$', fontsize=11)
plt.tight_layout()
plt.show()

## Detektionsgrænse: Hvornår er ghost synlig?

Jeg undersøger systematisk, for hvilke værdier af $m_{\text{ghost}}$ posterioren adskiller sig meningsfuldt fra prioren. Dette giver en *detektionsgrænse* for ghost-populationens styrke.

### Hypotese
- For $m_{\text{ghost}} < 0.3$ er den estimerede posterior nær-identisk med prioren (ghost-signalet er for svagt). Detektionsgrænsen er $m_{\text{ghost}} \approx 0.5$ for $n = 3000$ observationer.

In [ ]:
# Detektionsgrænse: posterior std på m_ghost som funktion af sand m_ghost
m_ghost_detect = [0.1, 0.3, 0.5, 1.0, 2.0, 4.0]
true_N_det    = 1.0
true_mobs_det = 1.0
N_OBS_det     = 3000
step_det = ExpStepSize(first_step=0.05, last_step=0.005, tau=40.0)

rows_detect = []
prior_std_ghost = GaussPrior(ci=[0.0, 5.0]).std  # prior std som reference

for m_g in m_ghost_detect:
    graph_g2.update_weights([true_N_det, true_mobs_det, m_g])
    obs_det = graph_g2.sample(N_OBS_det)

    sv_det = graph_g2.svgd(
        observed_data=obs_det,
        prior=[
            GaussPrior(ci=[0.2, 3.0]),
            GaussPrior(ci=[0.1, 5.0]),
            GaussPrior(ci=[0.0, 5.0]),
        ],
        n_particles=30,
        n_iter=200,
        step_size=step_det,
    )

    post_mean_ghost = sv_det.mean()[2]
    post_std_ghost  = sv_det.std()[2]
    rows_detect.append({
        'm_ghost (sand)':       m_g,
        'Posterior mean':       post_mean_ghost,
        'Posterior std':        post_std_ghost,
        'Bias':                 post_mean_ghost - m_g,
    })
    print(f"m_ghost={m_g:.1f}: post. mean={post_mean_ghost:.3f}, std={post_std_ghost:.3f}")

df_detect = pd.DataFrame(rows_detect)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(df_detect['m_ghost (sand)'], df_detect['Posterior mean'], 'o-', label='Posterior mean')
axes[0].plot(df_detect['m_ghost (sand)'], df_detect['m_ghost (sand)'], 'k--', label='Sand = estimat')
axes[0].fill_between(
    df_detect['m_ghost (sand)'],
    df_detect['Posterior mean'] - 1.96 * df_detect['Posterior std'],
    df_detect['Posterior mean'] + 1.96 * df_detect['Posterior std'],
    alpha=0.2, label='95% CI'
)
axes[0].set_xlabel('Sand $m_{\\mathrm{ghost}}$')
axes[0].set_ylabel('Estimeret $m_{\\mathrm{ghost}}$')
axes[0].set_title('Estimat vs. sand ghost-migrationsrate')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df_detect['m_ghost (sand)'], df_detect['Posterior std'], 's-', color='C1')
axes[1].set_xlabel('Sand $m_{\\mathrm{ghost}}$')
axes[1].set_ylabel('Posterior std på $m_{\\mathrm{ghost}}$')
axes[1].set_title('Posterior usikkerhed vs. sand ghost-migrationsrate')
axes[1].grid(alpha=0.3)

plt.suptitle(f'Detektionsgrænse for ghost-migration ($n={N_OBS_det}$ obs.)', fontsize=11)
plt.tight_layout()
plt.show()

## Prøvestørrelseeffekt: Hvornår er ghost-population detektabel?

Detektionsgrænsen afhænger af datasætstørrelsen. Jeg undersøger, hvordan posterior-bredden på $m_{\text{ghost}}$ falder med antallet af observationer for et fast sandt $m_{\text{ghost}}$.

### Hypotese
- Posterior std på $m_{\text{ghost}}$ falder som $\sim 1/\sqrt{n}$ som for de øvrige parametre, men udgangsniveauet er højere – det kræver 3–5 gange så mange observationer at estimere $m_{\text{ghost}}$ med samme relative præcision som $m_{\text{obs}}$.

In [ ]:
true_theta_ss_g = [1.0, 1.0, 1.0]  # m_ghost = 1.0 (detektabelt)
sample_sizes_g  = [200, 500, 1000, 2000, 5000, 10000]
step_ss_g = ExpStepSize(first_step=0.05, last_step=0.005, tau=40.0)

graph_g2.update_weights(true_theta_ss_g)
rows_ss_g = []

for n in sample_sizes_g:
    obs_n = graph_g2.sample(n)
    sv = graph_g2.svgd(
        observed_data=obs_n,
        prior=[
            GaussPrior(ci=[0.2, 3.0]),
            GaussPrior(ci=[0.1, 5.0]),
            GaussPrior(ci=[0.0, 5.0]),
        ],
        n_particles=25,
        n_iter=150,
        step_size=step_ss_g,
    )
    rows_ss_g.append({
        'n': n,
        '1/N std':    sv.std()[0],
        'm_obs std':  sv.std()[1],
        'm_ghost std': sv.std()[2],
    })
    print(f"n={n:6d}: m_ghost std={sv.std()[2]:.3f}")

df_ss_g = pd.DataFrame(rows_ss_g)
n_arr   = np.array(sample_sizes_g, dtype=float)

fig, ax = plt.subplots(figsize=(8, 5))
for col, label, color in [
    ('1/N std',     '$1/N$',            'C0'),
    ('m_obs std',   '$m_{\\mathrm{obs}}$',   'C1'),
    ('m_ghost std', '$m_{\\mathrm{ghost}}$', 'C2'),
]:
    ax.loglog(df_ss_g['n'], df_ss_g[col], 'o-', color=color, label=label)

# 1/sqrt(n) reference
ref_scale = df_ss_g['m_ghost std'].iloc[0] * np.sqrt(sample_sizes_g[0])
ax.loglog(n_arr, ref_scale / np.sqrt(n_arr), 'k--', alpha=0.5, label='$\\propto 1/\\sqrt{n}$')

ax.set_xlabel('Antal observationer $n$ (log-skala)')
ax.set_ylabel('Posterior std (log-skala)')
ax.set_title('Konvergens af parameterestimater i ghost-modellen')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()